# WAQ Solar Radiation Binary Files Comparison

Compare two Delft3D-WAQ binary input files for shortwave radiation flux.

In [1]:
import struct
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# File paths
file1 = "<MODEL_DIR>/02scen/waq-daily-sw_radiation_flux-02towaq_fpv41_nogaps_fixed.bin"
file2 = "<MODEL_DIR>/01scen_baseline/waq-sw_radiation_flux-noFPV_fixed.bin"

print(f"File 1: {Path(file1).name}")
print(f"File 2: {Path(file2).name}")

File 1: waq-daily-sw_radiation_flux-02towaq_fpv41_nogaps_fixed.bin
File 2: waq-sw_radiation_flux-noFPV_fixed.bin


In [2]:
# Get file sizes
size1 = Path(file1).stat().st_size
size2 = Path(file2).stat().st_size
print(f"Size File 1: {size1:,} bytes")
print(f"Size File 2: {size2:,} bytes")
print(f"Number of floats: {size1 // 4:,}")

Size File 1: 24,376,000 bytes
Size File 2: 24,376,000 bytes
Number of floats: 6,094,000


In [3]:
def read_floats(filename, max_floats=50000):
    """Read first N floats from binary file"""
    floats = []
    with open(filename, "rb") as f:
        for _ in range(max_floats):
            data = f.read(4)
            if not data:
                break
            floats.append(struct.unpack("f", data)[0])
    return np.array(floats)

# Read first chunk
data1 = read_floats(file1)
data2 = read_floats(file2)

print(f"Read {len(data1):,} floats from each file")

Read 50,000 floats from each file


In [4]:
# Find first difference
diff_idx = np.where(np.abs(data1 - data2) > 1e-5)[0]
if len(diff_idx) > 0:
    first_diff = diff_idx[0]
    print(f"First difference at float index {first_diff}:")
    print(f"  File1: {data1[first_diff]:.6f}")
    print(f"  File2: {data2[first_diff]:.6f}")
else:
    print("No differences found in first 50,000 floats!")

First difference at float index 121:
  File1: 102.843399
  File2: 112.705101


In [5]:
# Plot first 1000 values
plt.figure(figsize=(12, 6))
plt.plot(data1[:1000], label='Daily-averaged (FPV41)', alpha=0.8)
plt.plot(data2[:1000], label='No FPV', alpha=0.8)
plt.title('First 1000 Float Values Comparison')
plt.xlabel('Float Index')
plt.ylabel('Solar Radiation Value')
plt.legend()
plt.grid(True)
plt.show()

<Figure size 1200x600 with 1 Axes>

In [6]:
# Statistics
print("Statistics:")
print(f"File 1 - Mean: {data1.mean():.4f}, Min: {data1.min():.4f}, Max: {data1.max():.4f}")
print(f"File 2 - Mean: {data2.mean():.4f}, Min: {data2.min():.4f}, Max: {data2.max():.4f}")

Statistics:
File 1 - Mean: 25.3482, Min: 0.0000, Max: 251.7330
File 2 - Mean: 28.4804, Min: 0.0000, Max: 251.7330


## Full File Comparison (optional - slow for large files)

Uncomment if you want to check the entire files:

In [7]:
def compare_full_files():
    with open(file1, "rb") as f1, open(file2, "rb") as f2:
        idx = 0
        diff_count = 0
        while True:
            b1 = f1.read(4)
            b2 = f2.read(4)
            if not b1 or not b2:
                break
            v1 = struct.unpack("f", b1)[0]
            v2 = struct.unpack("f", b2)[0]
            if abs(v1 - v2) > 1e-5:
                if diff_count < 10:
                    print(f"Diff at {idx}: {v1:.6f} vs {v2:.6f}")
                diff_count += 1
            idx += 1
            if idx % 1000000 == 0:
                print(f"Processed {idx:,} floats...")
        print(f"Total differences: {diff_count:,}")

compare_full_files()

Diff at 121: 102.843399 vs 112.705101
Diff at 122: 102.843399 vs 112.705101
Diff at 146: 102.843399 vs 112.705101
Diff at 147: 92.981705 vs 112.705101
Diff at 148: 102.843399 vs 112.705101
Diff at 150: 102.843399 vs 112.705101
Diff at 151: 92.981705 vs 112.705101
Diff at 152: 92.981705 vs 112.705101
Diff at 153: 83.120010 vs 112.705101
Diff at 154: 83.120010 vs 112.705101
Processed 1,000,000 floats...
Processed 2,000,000 floats...
Processed 3,000,000 floats...
Processed 4,000,000 floats...
Processed 5,000,000 floats...
Processed 6,000,000 floats...
Total differences: 344,588
